In [16]:
import dill
import numpy as np
import time
import uuid
import random
import requests
from datetime import datetime

# Load the already-fitted model (fast, no refitting needed)
with open("kde_models_3d.pkl", "rb") as f:
    kde_models_3d = dill.load(f)

def generate_flight_inputs(fleet, max_attempts=50, rng=None):
    f = kde_models_3d[fleet]
    kde = f["kde"]
    rng = rng or np.random.default_rng()

    for _ in range(max_attempts):
        sample = kde.resample(1, seed=rng)
        distance = float(np.exp(sample[0, 0]))
        weight = float(np.exp(sample[1, 0]))
        tailwind = float(sample[2, 0])

        if (f["dist_min"] <= distance <= f["dist_max"] and
                f["weight_min"] <= weight <= f["weight_max"] and
                f["tailwind_min"] <= tailwind <= f["tailwind_max"]):
            return distance, weight, tailwind

    distance = min(max(distance, f["dist_min"]), f["dist_max"])
    weight = min(max(weight, f["weight_min"]), f["weight_max"])
    tailwind = min(max(tailwind, f["tailwind_min"]), f["tailwind_max"])
    return distance, weight, tailwind


# Generate flights continuously, sending each one to Django
count = 0
try:
    while True:
        rng = np.random.default_rng()
        fleet = random.choice(list(kde_models_3d.keys()))
        distance, weight, tailwind = generate_flight_inputs(fleet, rng=rng)

        record = {
            "flight_id": str(uuid.uuid4()),
            "fleet": fleet,
            "great_circle_distance_nm": round(distance, 1),
            "gross_weight_at_liftoff_kg": round(weight, 0),
            "max_tailwind_during_takeoff_kt": round(tailwind, 1),
            "generated_at": datetime.utcnow().isoformat(),
        }

        response = requests.post("http://localhost:8000/api/flights/", json=record)
        count += 1
        print(f"{count}: {response.status_code}  {fleet}  {distance:.0f} NM  {weight:.0f} kg  {tailwind:.1f} kt")

        time.sleep(2)
except KeyboardInterrupt:
    print(f"\nStopped after {count} flights")

1: 201  A330  2519 NM  178256 kg  0.5 kt
2: 201  B737F-NG  1449 NM  75761 kg  -9.2 kt
3: 201  B737F-NG  672 NM  63393 kg  -2.8 kt
4: 201  B737F-NG  2194 NM  63500 kg  -5.9 kt
5: 201  B737-NG  623 NM  64737 kg  -3.0 kt
6: 201  B737-NG  560 NM  55426 kg  3.8 kt
7: 201  B737F-NG  396 NM  59863 kg  4.5 kt
8: 201  B737-NG  228 NM  54283 kg  -9.5 kt
9: 201  B737-NG  1359 NM  61570 kg  -7.4 kt
10: 201  A330  3047 NM  181905 kg  6.7 kt
11: 201  A330  215 NM  155355 kg  -1.0 kt
12: 201  A330  156 NM  156227 kg  -6.3 kt
13: 201  B737-NG  227 NM  56287 kg  -12.1 kt
14: 201  B737F-NG  747 NM  61806 kg  -5.1 kt
15: 201  A330  4122 NM  222014 kg  9.0 kt
16: 201  A330  165 NM  161738 kg  -7.8 kt
17: 201  B737-NG  280 NM  62427 kg  -4.1 kt
18: 201  B737F-NG  325 NM  58562 kg  -5.0 kt
19: 201  B737F-NG  360 NM  60267 kg  -3.4 kt
20: 201  B737-NG  1259 NM  67192 kg  -3.6 kt
21: 201  A330  2834 NM  206942 kg  1.0 kt
22: 201  A330  1761 NM  196373 kg  3.1 kt
23: 201  A330  169 NM  145791 kg  0.7 kt
24: 20